# Fase 6 (a) — Ablación: ¿ayudan los chips con etiqueta débil?  (Colab)

**Entregable 2(a):** entrenar solo con los chips dorados vs. añadir los 4,384 chips con etiqueta débil (Otsu sobre S1).
Todas las variantes usan la **pérdida combinada** (ganadora de la ablación (b)) y el mismo presupuesto (60 épocas, paciencia 15).

| Variante | Entrenamiento | Pregunta |
|---|---|---|
| `solo_dorados` | 252 dorados | referencia (= `combinada` de la fase 6b) |
| `solo_debiles` | solo débiles (30 épocas × 1000 chips) | ¿qué aprende la red **solo** de etiquetas ruidosas? ¿supera a Otsu, que las generó? |
| `mezcla` | 252 dorados + 750 débiles aleatorios por época | ¿más datos (aunque ruidosos) ayudan desde el inicio? |
| `preentreno_ajuste` | `solo_debiles` → ajuste con 252 dorados | ¿las débiles sirven como *preentrenamiento* y las doradas corrigen su sesgo? |

**Cuidados metodológicos:**
- Se **excluyen** los chips débiles de **Bolivia** (región no vista del entregable 4) y los que coincidan con algún chip a mano.
- **Presupuesto controlado:** cada época usa una muestra aleatoria de débiles → corridas de minutos, no horas.
- **3 semillas**, comparación en **validación** (test intacto hasta la fase 7). Reanudable: las corridas terminadas se reutilizan.
- Recordatorio de la fase 1: Otsu tiene IoU ≈ 0.50 contra las etiquetas doradas, con mucha **comisión** en Somalia/Pakistán → las débiles enseñan a "inventar agua" en zonas áridas.

**Colab:** requiere GPU T4. Tiempo estimado **~2 horas** (+ descarga de ~9 GB al disco local de Colab, no a Drive).
Si la sesión se cae, *Ejecutar todo* de nuevo: las corridas terminadas se reutilizan. Si subiste a Drive los resultados de la fase 6b, las 3 corridas `combinada` **no se repiten**.

## 0. Preparación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
RAIZ_PROYECTO = Path('/content/drive/MyDrive/rnap_p2_inundaciones')
sys.path.insert(0, str(RAIZ_PROYECTO))

!pip install -q rasterio
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 1. Módulo nuevo de `src/`

In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/ablacion_debil.py
"""
ablacion_debil.py
Ablación (a): ¿ayudan los 4,384 chips con etiqueta DÉBIL (Otsu sobre S1)?
Estrategias (todas con la pérdida elegida en la ablación (b)):
  - solo_dorados      : solo los 252 chips dorados de train (ya calculada como 'combinada' en la fase 6b)
  - mezcla            : dorados + débiles juntos desde el inicio
  - preentreno_ajuste : 1) preentrenar solo con débiles  2) ajustar solo con dorados partiendo de esos pesos

Cuidados metodológicos:
  - Se EXCLUYEN los chips débiles de Bolivia (región de prueba no vista) y cualquier chip débil cuyo
    id coincida con un chip etiquetado a mano (evita fuga hacia valid/test y etiquetas contradictorias).
  - Presupuesto: en cada época se usa una MUESTRA aleatoria de débiles (no los 4,384), para que cada
    corrida dure minutos y no horas. A lo largo del entrenamiento cada chip débil se ve varias veces.
"""
import gc
import json
from dataclasses import replace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import ConcatDataset, DataLoader, Sampler

from . import entrenamiento as _entrenamiento
from .cargadores import crear_cargadores
from .configuracion import DIR_S1_DEBIL, DIR_ETIQUETA_DEBIL, VALOR_SIN_DATO
from .conjunto_datos import ConjuntoInundaciones
from .entrenamiento import entrenar, cargar_mejor_modelo
from .evaluacion import evaluar
from .lectura import leer_todas_las_particiones
from .utilidades import fijar_semilla

SUFIJO_S1_DEBIL, SUFIJO_ETIQUETA_DEBIL = "S1Weak", "S1OtsuLabelWeak"


# ----------------------------------------------------------------------------- datos débiles
def listar_chips_debiles() -> pd.DataFrame:
    """ids de chips débiles utilizables (con imagen y etiqueta), sin Bolivia ni ids de chips a mano."""
    ids_imagen = {r.name.replace(f"_{SUFIJO_S1_DEBIL}.tif", "") for r in DIR_S1_DEBIL.glob("*.tif")}
    ids_etiqueta = {r.name.replace(f"_{SUFIJO_ETIQUETA_DEBIL}.tif", "") for r in DIR_ETIQUETA_DEBIL.glob("*.tif")}
    ids = sorted(ids_imagen & ids_etiqueta)
    if not ids:
        ejemplo = [r.name for r in list(DIR_S1_DEBIL.glob("*"))[:3]]
        raise RuntimeError(f"No se emparejaron imágenes y etiquetas débiles. Ejemplos de archivos: {ejemplo}")
    df = pd.DataFrame({"id_chip": ids})
    df["evento"] = df["id_chip"].str.split("_").str[0]
    ids_mano = set(leer_todas_las_particiones()["id_chip"])
    df["excluido"] = np.select([df["evento"] == "Bolivia", df["id_chip"].isin(ids_mano)],
                               ["Bolivia (región no vista)", "coincide con chip a mano"], default="")
    return df


class ConjuntoDebil(ConjuntoInundaciones):
    """Igual que el conjunto dorado, pero lee S1Weak / S1OtsuLabelWeak desde disco (no caben en RAM)
    y sanea la etiqueta: cualquier valor fuera de {0, 1} se marca como -1 (ignorar)."""

    def __init__(self, ids_chips, normalizador, **parametros):
        super().__init__(ids_chips, normalizador, en_memoria=False,
                         dir_s1=DIR_S1_DEBIL, sufijo_s1=SUFIJO_S1_DEBIL,
                         dir_etiqueta=DIR_ETIQUETA_DEBIL, sufijo_etiqueta=SUFIJO_ETIQUETA_DEBIL, **parametros)

    def _cargar(self, id_chip):
        imagen, etiqueta = super()._cargar(id_chip)
        etiqueta[(etiqueta != 0) & (etiqueta != 1)] = VALOR_SIN_DATO
        return imagen, etiqueta


class MuestreadorMixto(Sampler):
    """Cada época: TODOS los índices dorados + una muestra aleatoria (sin reemplazo) de n débiles, barajados."""

    def __init__(self, n_dorados: int, n_debiles_total: int, n_debiles_por_epoca: int, semilla: int):
        self.n_dorados, self.n_debiles_total = n_dorados, n_debiles_total
        self.n_debiles_por_epoca = min(n_debiles_por_epoca, n_debiles_total)
        self.generador = torch.Generator().manual_seed(semilla)

    def __len__(self):
        return self.n_dorados + self.n_debiles_por_epoca

    def __iter__(self):
        debiles = self.n_dorados + torch.randperm(self.n_debiles_total, generator=self.generador)[:self.n_debiles_por_epoca]
        indices = torch.cat([torch.arange(self.n_dorados), debiles])
        return iter(indices[torch.randperm(len(indices), generator=self.generador)].tolist())


def _semilla_por_worker_mixto(id_worker: int):
    """Como semilla_por_worker, pero propagando el generador a cada sub-conjunto del ConcatDataset."""
    informacion = torch.utils.data.get_worker_info()
    semilla = (torch.initial_seed() + id_worker) % 2**32
    for subconjunto in informacion.dataset.datasets:
        subconjunto.generador = np.random.default_rng(semilla)


def crear_cargador_mixto(conjunto_dorado, ids_debiles, normalizador, n_debiles_por_epoca: int,
                         incluir_dorados: bool, tamano_lote: int = 8, tamano_recorte: int = 256,
                         num_workers: int = 2, semilla: int = 42) -> DataLoader:
    debiles = ConjuntoDebil(ids_debiles, normalizador, aumentar_datos=True,
                            tamano_recorte=tamano_recorte, semilla=semilla)
    dorados = conjunto_dorado if incluir_dorados else ConjuntoInundaciones([], normalizador)
    conjunto = ConcatDataset([dorados, debiles])
    muestreador = MuestreadorMixto(len(dorados), len(debiles), n_debiles_por_epoca, semilla)
    return DataLoader(conjunto, batch_size=tamano_lote, sampler=muestreador, drop_last=True,
                      num_workers=num_workers, pin_memory=torch.cuda.is_available(),
                      worker_init_fn=_semilla_por_worker_mixto, persistent_workers=num_workers > 0)


# ----------------------------------------------------------------------------- entrenamiento
def entrenar_desde_pesos(configuracion, ruta_pesos, *argumentos, **parametros):
    """Llama a entrenar() pero con el modelo INICIALIZADO desde un checkpoint (ajuste fino).
    Se sustituye temporalmente crear_modelo para no tocar entrenamiento.py."""
    crear_original = _entrenamiento.crear_modelo

    def crear_con_pesos(cfg):
        modelo = crear_original(cfg)
        estado = torch.load(ruta_pesos, map_location="cpu", weights_only=False)
        modelo.load_state_dict(estado["modelo"])
        return modelo

    _entrenamiento.crear_modelo = crear_con_pesos
    try:
        return entrenar(configuracion, *argumentos, **parametros)
    finally:
        _entrenamiento.crear_modelo = crear_original


def _entrenar_si_falta(dir_corrida, funcion, *argumentos, **parametros):
    if (dir_corrida / "resumen.json").exists():
        print(f"  {dir_corrida.name}: ya estaba terminada, se reutiliza")
        return
    funcion(*argumentos, **parametros)
    (dir_corrida / "ultimo.pt").unlink(missing_ok=True)


def _evaluar_y_guardar(dir_corrida, cargador_valid, dispositivo):
    modelo, _, mejor_epoca = cargar_mejor_modelo(dir_corrida, dispositivo)
    acumulador, _ = evaluar(modelo, cargador_valid, dispositivo)
    acumulador.tabla_por_evento().to_csv(dir_corrida / "por_evento_valid.csv")
    with open(dir_corrida / "metricas_valid.json", "w") as f:
        json.dump(acumulador.metricas(), f, indent=2)
    del modelo
    return mejor_epoca, acumulador.metricas()


def correr_ablacion_debil(configuracion_base, semillas, normalizador, ids_debiles, dispositivo, dir_experimentos,
                          debiles_mezcla: int = 750, debiles_preentreno: int = 1000, epocas_preentreno: int = 30,
                          tamano_lote: int = 8, tamano_recorte: int = 256, num_workers: int = 2) -> pd.DataFrame:
    filas = []
    for semilla in semillas:
        fijar_semilla(semilla)
        cargadores = crear_cargadores(normalizador, tamano_lote, tamano_recorte, num_workers, semilla)
        comunes = dict(tamano_lote=tamano_lote, tamano_recorte=tamano_recorte, num_workers=num_workers, semilla=semilla)

        # --- mezcla: dorados + débiles desde el inicio
        configuracion = replace(configuracion_base, nombre=f"mezcla_s{semilla}", semilla=semilla,
                                fuente_entrenamiento="mano+debil")
        dir_corrida = dir_experimentos / configuracion.nombre
        print(f"\n{'=' * 70}\n▶ {configuracion.nombre}  (252 dorados + {debiles_mezcla} débiles por época)\n{'=' * 70}")
        cargador = crear_cargador_mixto(cargadores["train"].dataset, ids_debiles, normalizador,
                                        debiles_mezcla, incluir_dorados=True, **comunes)
        _entrenar_si_falta(dir_corrida, entrenar, configuracion, cargador, cargadores["valid"], dispositivo, dir_corrida)
        filas.append({"variante": "mezcla", "semilla": semilla,
                      **dict(zip(["mejor_epoca", "m"], _evaluar_y_guardar(dir_corrida, cargadores["valid"], dispositivo)))})

        # --- preentreno (solo débiles) -> ajuste (solo dorados)
        configuracion_pre = replace(configuracion_base, nombre=f"preentreno_s{semilla}", semilla=semilla,
                                    fuente_entrenamiento="debil", epocas=epocas_preentreno,
                                    paciencia=max(5, configuracion_base.paciencia * epocas_preentreno // configuracion_base.epocas))
        dir_pre = dir_experimentos / configuracion_pre.nombre
        print(f"\n{'=' * 70}\n▶ {configuracion_pre.nombre}  ({debiles_preentreno} débiles por época)\n{'=' * 70}")
        cargador = crear_cargador_mixto(cargadores["train"].dataset, ids_debiles, normalizador,
                                        debiles_preentreno, incluir_dorados=False, **comunes)
        _entrenar_si_falta(dir_pre, entrenar, configuracion_pre, cargador, cargadores["valid"], dispositivo, dir_pre)

        configuracion_ajuste = replace(configuracion_base, nombre=f"preentreno_ajuste_s{semilla}", semilla=semilla,
                                       fuente_entrenamiento="debil->mano")
        dir_corrida = dir_experimentos / configuracion_ajuste.nombre
        print(f"\n{'=' * 70}\n▶ {configuracion_ajuste.nombre}  (ajuste con 252 dorados)\n{'=' * 70}")
        _entrenar_si_falta(dir_corrida, entrenar_desde_pesos, configuracion_ajuste, dir_pre / "mejor.pt",
                           cargadores["train"], cargadores["valid"], dispositivo, dir_corrida)
        filas.append({"variante": "preentreno_ajuste", "semilla": semilla,
                      **dict(zip(["mejor_epoca", "m"], _evaluar_y_guardar(dir_corrida, cargadores["valid"], dispositivo)))})
        # también el modelo SOLO preentrenado (útil para el informe: ¿qué aprende de etiquetas ruidosas?)
        filas.append({"variante": "solo_debiles", "semilla": semilla,
                      **dict(zip(["mejor_epoca", "m"], _evaluar_y_guardar(dir_pre, cargadores["valid"], dispositivo)))})

        del cargadores, cargador
        gc.collect(); torch.cuda.empty_cache()

    df = pd.DataFrame(filas)
    return pd.concat([df.drop(columns="m"), pd.DataFrame(list(df["m"]))], axis=1)


## 2. (Colab ya tiene en Drive los archivos de fases anteriores)

In [ ]:
import pandas as pd
from src.configuracion import *

## 3. Imports y descarga (dorados + débiles)

In [ ]:
import torch
from src.utilidades import obtener_dispositivo
from src.descarga import descargar_mano, descargar_debil
from src.preprocesamiento import Normalizador
from src.experimento import ConfiguracionExperimento
from src.ablacion import correr_ablacion, resumir, por_evento_promedio
from src.ablacion_debil import listar_chips_debiles, ConjuntoDebil, correr_ablacion_debil
from src.visualizacion_ablacion import figura_ablacion

dispositivo = obtener_dispositivo()
assert dispositivo.type == 'cuda', 'Esta fase necesita GPU'
descargar_mano()
descargar_debil()          # ~9 GB, varios minutos
normalizador = Normalizador.desde_json(DIR_RESULTADOS / 'f1_estadisticas_normalizacion.json')
linea_base_otsu = pd.read_csv(DIR_RESULTADOS / 'f4_linea_base_otsu.csv', index_col=0)

## 4. Inspección de los chips débiles
**Qué verificar:** cuántos se excluyen (Bolivia y coincidencias con chips a mano), que la etiqueta tenga solo {−1, 0, 1}, y cómo se compara el % de agua con el de los dorados (≈ 9.4 %).

In [ ]:
df_debiles = listar_chips_debiles()
print(f"Chips débiles encontrados: {len(df_debiles)}")
display(pd.crosstab(df_debiles['evento'], df_debiles['excluido'].replace('', 'USABLE'), margins=True, margins_name='Total'))
ids_debiles = df_debiles.loc[df_debiles['excluido'] == '', 'id_chip'].tolist()
print(f"Chips débiles usables: {len(ids_debiles)}")

# Valores de etiqueta y % de agua en una muestra (comparar con ~9.4 % de los dorados de train)
muestra = ConjuntoDebil(pd.Series(ids_debiles).sample(40, random_state=SEMILLA).tolist(), normalizador)
etiquetas = torch.stack([muestra[i][1] for i in range(len(muestra))])
validos = etiquetas != VALOR_SIN_DATO
print('Valores de etiqueta débil:', sorted(etiquetas.unique().tolist()))
print(f"% agua (válidos) en 40 débiles: {100 * (etiquetas[validos] == 1).float().mean():.1f}%  |  % ignorados: {100 * (~validos).float().mean():.1f}%")

## 5. Configuración común

In [ ]:
from dataclasses import replace

base = ConfiguracionExperimento(perdida='combinada', epocas=60, paciencia=15)
SEMILLAS = [42, 7, 123]
DIR_EXPERIMENTOS = DIR_RESULTADOS / 'experimentos'

## 6a. Referencia: solo dorados (pérdida combinada)

In [ ]:
# Misma configuración y nombres que la variante 'combinada' de la fase 6b -> si ya existen, se reutilizan
df_solo_dorados = correr_ablacion({'combinada': base}, SEMILLAS, normalizador, dispositivo, DIR_EXPERIMENTOS,
                                  tamano_lote=8, tamano_recorte=256, num_workers=2)
df_solo_dorados['variante'] = 'solo_dorados'
df_solo_dorados.round(4)

## 6b. Variantes con débiles: mezcla, preentreno → ajuste (y solo débiles)

In [ ]:
df_debil = correr_ablacion_debil(base, SEMILLAS, normalizador, ids_debiles, dispositivo, DIR_EXPERIMENTOS,
                                 debiles_mezcla=750, debiles_preentreno=1000, epocas_preentreno=30,
                                 tamano_lote=8, tamano_recorte=256, num_workers=2)
df_debil.round(4)

## 7. Resumen: media ± desviación (3 semillas)

In [ ]:
orden = ['solo_dorados', 'solo_debiles', 'mezcla', 'preentreno_ajuste']
df_corridas = pd.concat([df_solo_dorados, df_debil], ignore_index=True)
df_corridas['variante'] = pd.Categorical(df_corridas['variante'], orden, ordered=True)
df_corridas = df_corridas.sort_values(['variante', 'semilla']).reset_index(drop=True)
df_corridas['variante'] = df_corridas['variante'].astype(str)
df_corridas.to_csv(DIR_RESULTADOS / 'f6a_ablacion_corridas.csv', index=False)

tabla_resumen = resumir(df_corridas)
tabla_resumen.to_csv(DIR_RESULTADOS / 'f6a_ablacion_resumen.csv')
tabla_resumen

## 8. Figura

In [ ]:
figura_ablacion(df_corridas, linea_base_otsu.loc['valid', 'iou'],
                'Ablación (a) — dorados vs. débiles (validación, 3 semillas)', DIR_FIGURAS / 'f6a_ablacion.png');

## 9. IoU por evento (promedio de semillas)

In [ ]:
carpetas = {'solo_dorados': 'combinada', 'solo_debiles': 'preentreno', 'mezcla': 'mezcla', 'preentreno_ajuste': 'preentreno_ajuste'}
tabla_eventos = por_evento_promedio(list(carpetas.values()), SEMILLAS, DIR_EXPERIMENTOS)
tabla_eventos.columns = list(carpetas.keys())
tabla_eventos.to_csv(DIR_RESULTADOS / 'f6a_ablacion_por_evento.csv')
tabla_eventos.round(3)

## Para traer a la conversación
La tabla de inspección de débiles (sección 4), el resumen media ± desv. (7), la figura (8) y la tabla por evento (9).
Con eso fijamos el **modelo final** que irá a test (fase 7), Bolivia/Lima (fase 8) y el análisis de fallos (fase 9).